In [1]:
import numpy as np
import pandas as pd
from obspy.core import read, UTCDateTime
from obspy.core.inventory import read_inventory
from obspy.core.util.attribdict import AttribDict
from obspy.clients.fdsn import Client
from collections import defaultdict
from obspy.geodetics import gps2dist_azimuth
from obspy.geodetics import kilometers2degrees
from obspy.taup import TauPyModel

In [2]:
# AUSPASS     https://auspass.edu.au
# BGR         https://eida.bgr.de
# BGS         https://eida.bgs.ac.uk
# EARTHSCOPE  https://service.earthscope.org
# EIDA        http://eida-federator.ethz.ch
# EMSC        https://www.seismicportal.eu
# EPOSFR      https://seisdata.epos-france.fr
# ETH         https://eida.ethz.ch
# GEOFON      https://geofon.gfz.de
# GEONET      https://service.geonet.org.nz
# GFZ         https://geofon.gfz.de
# ICGC        https://ws.icgc.cat
# IESDMC      http://batsws.earth.sinica.edu.tw
# IGN         http://fdsnws.sismologia.ign.es
# INGV        https://webservices.ingv.it
# IPGP        https://ws.ipgp.fr
# IRIS        https://service.earthscope.org
# IRISDMC     https://service.earthscope.org
# IRISPH5     https://service.earthscope.org
# ISC         https://www.isc.ac.uk
# KAGSR       http://sdis.emsd.ru
# KNMI        https://rdsa.knmi.nl
# KOERI       https://eida.koeri.boun.edu.tr
# LMU         https://erde.geophysik.uni-muenchen.de
# NCEDC       https://service.ncedc.org
# NIEP        https://eida-sc3.infp.ro
# NOA         https://eida.gein.noa.gr
# NRCAN       https://earthquakescanada.nrcan.gc.ca
# ODC         https://www.orfeus-eu.org
# ORFEUS      https://www.orfeus-eu.org
# RASPISHAKE  https://data.raspberryshake.org
# RESIF       https://ws.resif.fr
# SCEDC       https://service.scedc.caltech.edu
# TEXNET      http://rtserve.beg.utexas.edu
# UIB-NORSAR  https://eida.geo.uib.no
# USGS        https://earthquake.usgs.gov
# USP         https://sismo.iag.usp.br

In [3]:
def to_number(polarity):
    if polarity == 'positive':
        return 1
    elif polarity == 'negative':
        return -1
    else:
        return 0

In [4]:
# Set Client for Downloading Waveforms
client = Client("IRIS")
# client = Client("USGS")

In [5]:
# Select Catalog around time of large event in New Zealand
# Start time is P wave arrival time according to Wilber3

starttime = UTCDateTime(1994,1,17,0,0,0)
endtime = UTCDateTime(1994,1,18,0,0,0)
catalog = client.get_events(starttime=starttime, endtime=endtime, minmagnitude=6.0,
                        maxmagnitude=7.0, includearrivals=True)
print(catalog)

1 Event(s) in Catalog:
1994-01-17T12:30:54.620000Z | +34.136, -118.583 | 6.7  mw


In [7]:
# 6.7 magnitude in the catalog
# get all stations that recorded the event
event1 = catalog[0]
event1

Event:	1994-01-17T12:30:54.620000Z | +34.136, -118.583 | 6.7  mw

	            resource_id: ResourceIdentifier(id="smi:service.iris.edu/fdsnws/event/1/query?eventid=350828")
	             event_type: 'earthquake'
	    preferred_origin_id: ResourceIdentifier(id="smi:service.iris.edu/fdsnws/event/1/query?originid=2211176")
	 preferred_magnitude_id: ResourceIdentifier(id="smi:service.iris.edu/fdsnws/event/1/query?magnitudeid=14464897")
	                   ---------
	     event_descriptions: 1 Elements
	                origins: 1 Elements
	             magnitudes: 1 Elements

In [9]:
[pick for pick in event1.picks]

[]

In [ ]:
# create a dictionary that stores sign for each station
data_dict = defaultdict(list)

for idx, pick in enumerate(event1.picks):
    if idx % 10 == 0:
        print(f"Processing pick {idx}/{len(event1.picks)}")
    if pick.phase_hint == 'P':
        stat = pick.waveform_id.station_code
        net = pick.waveform_id.network_code
        chan = pick.waveform_id.channel_code
        loc = pick.waveform_id.location_code
        seed_id = '.'.join([net, stat, loc, chan])
        try:
            inventory = client.get_stations(starttime=pick.time, endtime=pick.time + 1,
                                            network=net, station=stat, channel=chan, level="channel")
        except Exception as e:
            print(f"Error occurred while fetching station information for {seed_id}")
            # print error message
            print(f"Error: {e}")
            continue
        
        try:
            coordinates = inventory.get_coordinates(seed_id)
        except Exception as e:
            print(f"Error occurred while fetching coordinates for {seed_id}")
            # print error message
            print(f"Error: {e}")
            continue
        
        if len(data_dict[seed_id]) == 0:
            data_dict[seed_id].extend(list(coordinates.values())[:2] + [to_number(pick.polarity)])
        
        

# collapse to sign
for seed_id in data_dict:
    if abs(data_dict[seed_id][2]) > 1: print(f"{seed_id} has polarity {data_dict[seed_id][2]}")
    data_dict[seed_id][2] = np.sign(data_dict[seed_id][2])

Processing pick 0/353
Error occurred while fetching station information for CI.MWC.N1.EHZ
Error: No data available for request.
HTTP Status code: 204
Detailed response of server:


Error occurred while fetching station information for CI.FTC.N2.EHZ
Error: No data available for request.
HTTP Status code: 204
Detailed response of server:


Error occurred while fetching coordinates for CI.SUN.N1.EHZ
Error: No matching channel metadata found.
Error occurred while fetching coordinates for CI.RYS.N1.EHZ
Error: No matching channel metadata found.
Error occurred while fetching coordinates for CI.ABL.N1.EHZ
Error: No matching channel metadata found.
Error occurred while fetching station information for CI.PLE.N1.EHZ
Error: No data available for request.
HTTP Status code: 204
Detailed response of server:


Error occurred while fetching station information for CI.BMT.N1.EHZ
Error: No data available for request.
HTTP Status code: 204
Detailed response of server:


Error occurred while fetching coo

In [189]:
# NOTE: get event location
lat = event1.origins[0].latitude
lon = event1.origins[0].longitude
hdepth = event1.origins[0].depth/1000 # convert to km

col_string = 'event_id, station, network, location, channel, p_polarity, takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty'
columns = [col.strip() for col in col_string.split(',')]
df = pd.DataFrame(columns=columns)
event_id = 1
takeoff_unc = 0.1 # placeholder
az_unc = 0.1 # placeholder

velocity_model = TauPyModel(model='ak135')

for key, value in data_dict.items():
    net, stat, loc, chan = key.split('.')
    p_polarity = value[2]
    src2dst = [lat, lon, value[0], value[1]]
    dist, az, baz = gps2dist_azimuth(*src2dst)
    epdist = kilometers2degrees(dist / 1000) # convert to km
    
    try:
        p_arrivals = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                            distance_in_degree=epdist, phase_list=['P'])
    except Exception as e:
        print(f"Error occurred while calculating travel times for station {stat}")
        # print error message
        print(f"Error: {e}")
        continue
    
    if len(p_arrivals) == 0:
        print(f"Station {stat} has no P arrival. Skipping.")
        print(f"Epidist = {epdist:.2f} degrees, depth = {hdepth} km")
        continue
    
    takeoff = p_arrivals[0].takeoff_angle
    
    if p_polarity != 0:
        df.loc[len(df)] = [event_id, stat, net, loc, chan, float(p_polarity),
                        takeoff, takeoff_unc, az, az_unc]
    else: print(f"Station {stat} has zero polarity. Skipping.")

Station LA00 has no P arrival. Skipping.
Epidist = 0.13 degrees, depth = 18.202 km
Station LA00 has no P arrival. Skipping.
Epidist = 0.13 degrees, depth = 18.202 km
Station LA02 has no P arrival. Skipping.
Epidist = 0.18 degrees, depth = 18.202 km
Station LA04 has no P arrival. Skipping.
Epidist = 0.35 degrees, depth = 18.202 km
Station PAS has no P arrival. Skipping.
Epidist = 0.31 degrees, depth = 18.202 km
Station USC has no P arrival. Skipping.
Epidist = 0.28 degrees, depth = 18.202 km
Station PAS has no P arrival. Skipping.
Epidist = 0.31 degrees, depth = 18.202 km
Station IR2 has no P arrival. Skipping.
Epidist = 0.21 degrees, depth = 18.202 km
Station PAS has no P arrival. Skipping.
Epidist = 0.31 degrees, depth = 18.202 km
Station TCC has zero polarity. Skipping.
Station SHH has zero polarity. Skipping.
Station PTD has no P arrival. Skipping.
Epidist = 0.31 degrees, depth = 18.202 km
Station WWR has zero polarity. Skipping.
Station GAV has zero polarity. Skipping.
Station SS2 

In [190]:
df

,event_id,station,network,location,channel,p_polarity,takeoff,takeoff_uncertainty,azimuth,azimuth_uncertainty
0,1,BAR,CI,,BHE,1.0,46.006038,0.1,133.920430,0.1
1,1,DGR,CI,,BHE,-1.0,46.007087,0.1,113.425264,0.1
2,1,GSC,CI,,BHE,-1.0,46.004644,0.1,52.201480,0.1
3,1,SBC,CI,,BHE,-1.0,63.126856,0.1,283.451126,0.1
4,1,SVD,CI,,BHE,1.0,46.008243,0.1,94.685583,0.1
...,...,...,...,...,...,...,...,...,...,...
124,1,PTQ,NC,,EHZ,1.0,46.005056,0.1,282.897540,0.1
125,1,PMG,NC,,EHZ,-1.0,46.006948,0.1,307.214136,0.1
126,1,WIN,CI,,EHZ,1.0,46.007603,0.1,113.525249,0.1
127,1,BAC,CI,,EHZ,1.0,46.007132,0.1,115.294783,0.1


In [191]:
# save to csv
df.to_csv('event1_data.csv', index=False)